In [2]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Working directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

required_files = [
    "sample1.tsv",
    "sample2.tsv",
    "sample3.tsv"
]

print("\nChecking input files...")

for filename in required_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        print(f"✓ Found: {filename}")
    else:
        print(f"✗ MISSING: {filename}")
        raise FileNotFoundError(
            f"Could not find {filename}\n"
            f"Expected location: {file_path}"
        )


def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    if not text:
        return []

    return text.split()


COMMON_TOKENS = {
    "the",
    "and",
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "company",
    "co",
    "ltd",
    "limited",
    "llc",
    "plc",
    "pvt",
    "private",
    "services",
    "technology",
    "technologies",
    "india",
    "usa",
    "us"
}


def prepare_dataframe(df):
    df = df.copy()

    df["name_normalized"] = (
        df["business_name"]
        .fillna("")
        .apply(normalize_text)
    )

    df["address_normalized"] = (
        df["business_address"]
        .fillna("")
        .apply(normalize_text)
    )

    df["country_normalized"] = (
        df["country"]
        .fillna("")
        .apply(normalize_text)
    )

    df["combined_normalized"] = (
        df["name_normalized"]
        + " "
        + df["address_normalized"]
    )

    df["name_tokens"] = (
        df["name_normalized"]
        .apply(tokenize)
    )

    df["address_tokens"] = (
        df["address_normalized"]
        .apply(tokenize)
    )

    return df


def build_token_index(df):
    token_index = defaultdict(set)

    for _, row in df.iterrows():
        entity_id = row["entity_id"]

        tokens = set(
            row["name_tokens"]
            + row["address_tokens"]
        )

        for token in tokens:
            if token in COMMON_TOKENS:
                continue

            if len(token) < 2:
                continue

            token_index[token].add(entity_id)

    return token_index


def build_country_index(df):
    country_index = defaultdict(set)

    for _, row in df.iterrows():
        country = row["country_normalized"]

        if country:
            country_index[country].add(
                row["entity_id"]
            )

    return country_index


def generate_candidates(
    source1_row,
    token_index,
    country_index
):
    candidates = set()

    name_tokens = set(
        source1_row["name_tokens"]
    )

    address_tokens = set(
        source1_row["address_tokens"]
    )

    all_tokens = name_tokens | address_tokens

    for token in all_tokens:
        if token in COMMON_TOKENS:
            continue

        if len(token) < 2:
            continue

        matching_ids = token_index.get(
            token,
            set()
        )

        candidates.update(
            matching_ids
        )

    country = source1_row[
        "country_normalized"
    ]

    if country:
        country_candidates = country_index.get(
            country,
            set()
        )

        if not candidates:
            candidates.update(
                country_candidates
            )

    return candidates


def generate_source_candidates(
    source1_df,
    target_df
):
    print("\nBuilding token index...")

    token_index = build_token_index(
        target_df
    )

    print(
        f"Token index entries: "
        f"{len(token_index)}"
    )

    print("Building country index...")

    country_index = build_country_index(
        target_df
    )

    print(
        f"Country index entries: "
        f"{len(country_index)}"
    )

    candidate_pairs = []

    for _, s1_row in source1_df.iterrows():
        s1_id = s1_row["entity_id"]

        candidates = generate_candidates(
            s1_row,
            token_index,
            country_index
        )

        for candidate_id in candidates:
            candidate_pairs.append(
                (
                    s1_id,
                    candidate_id
                )
            )

    return candidate_pairs


def generate_candidate_pairs(
    source1_df,
    source2_df,
    source3_df
):
    print("\n========================================")
    print("Generating Source 2 candidates")
    print("========================================")

    pairs_s2 = generate_source_candidates(
        source1_df,
        source2_df
    )

    print(
        f"Source 2 candidate pairs: "
        f"{len(pairs_s2)}"
    )

    print("\n========================================")
    print("Generating Source 3 candidates")
    print("========================================")

    pairs_s3 = generate_source_candidates(
        source1_df,
        source3_df
    )

    print(
        f"Source 3 candidate pairs: "
        f"{len(pairs_s3)}"
    )

    all_pairs = pairs_s2 + pairs_s3
    all_pairs = set(all_pairs)

    candidate_dict = defaultdict(set)

    for s1_id, candidate_id in all_pairs:
        candidate_dict[s1_id].add(
            candidate_id
        )

    for s1_id in source1_df["entity_id"]:
        if s1_id not in candidate_dict:
            candidate_dict[s1_id] = set()

    return candidate_dict


def save_candidate_pairs(
    candidate_dict,
    output_path
):
    rows = []

    for s1_id in sorted(
        candidate_dict.keys()
    ):
        candidates = sorted(
            candidate_dict[s1_id]
        )

        candidate_string = ",".join(
            candidates
        )

        rows.append(
            {
                "source1_entity_id": s1_id,
                "candidate_entity_ids": candidate_string
            }
        )

    output_df = pd.DataFrame(rows)

    output_df.to_csv(
        output_path,
        sep="\t",
        index=False
    )

    print("\nCandidate file saved:")
    print(output_path)


def print_sample_results(
    candidate_dict
):
    print("\n========================================")
    print("SAMPLE CANDIDATE RESULTS")
    print("========================================")

    count = 0

    for s1_id in sorted(
        candidate_dict.keys()
    ):
        candidates = sorted(
            candidate_dict[s1_id]
        )

        print(
            f"{s1_id} -> {candidates}"
        )

        count += 1

        if count >= 10:
            break


def main():
    # print("\n")
    # print("========================================")
    # print(" BUSINESS ENTITY RESOLUTION")
    # print(" STEP 2 - BLOCKING")
    # print("========================================")

    print("\nReading input files...")

    source1 = pd.read_csv(
        DATA_DIR / "sample1.tsv",
        sep="\t"
    )

    source2 = pd.read_csv(
        DATA_DIR / "sample2.tsv",
        sep="\t"
    )

    source3 = pd.read_csv(
        DATA_DIR / "sample3.tsv",
        sep="\t"
    )

    print(
        f"Source 1 records: {len(source1)}"
    )

    print(
        f"Source 2 records: {len(source2)}"
    )

    print(
        f"Source 3 records: {len(source3)}"
    )

    print("\nSource 1 columns:")
    print(source1.columns.tolist())

    print("\nSource 2 columns:")
    print(source2.columns.tolist())

    print("\nSource 3 columns:")
    print(source3.columns.tolist())

    print("\nPreparing data...")

    source1 = prepare_dataframe(source1)
    source2 = prepare_dataframe(source2)
    source3 = prepare_dataframe(source3)

    print("✓ Data preparation complete")

    candidate_dict = generate_candidate_pairs(
        source1,
        source2,
        source3
    )

    print_sample_results(
        candidate_dict
    )

    output_path = (
        OUTPUT_DIR
        / "candidate_pairs.tsv"
    )

    save_candidate_pairs(
        candidate_dict,
        output_path
    )

    total_pairs = sum(
        len(candidates)
        for candidates
        in candidate_dict.values()
    )

    total_s1 = len(source1)

    if total_s1 > 0:
        average_candidates = (
            total_pairs / total_s1
        )
    else:
        average_candidates = 0

    print("\n========================================")
    print(" BLOCKING COMPLETE")
    print("========================================")

    print(
        f"Source 1 entities: {total_s1}"
    )

    print(
        f"Total candidate pairs: {total_pairs}"
    )

    print(
        f"Average candidates per S1: "
        f"{average_candidates:.2f}"
    )

    print("\nOutput file:")
    print(output_path)

    print("\nDone! 🎉")


main()

Working directory: /home/ec2-user/SageMaker/amazon-ml-challenge/dotENV_submission/code/fetch_all
Data directory: /home/ec2-user/SageMaker/amazon-ml-challenge/dotENV_submission/code/fetch_all/data
Output directory: /home/ec2-user/SageMaker/amazon-ml-challenge/dotENV_submission/code/fetch_all/output

Checking input files...
✓ Found: sample1.tsv
✓ Found: sample2.tsv
✓ Found: sample3.tsv


 BUSINESS ENTITY RESOLUTION
 STEP 2 - BLOCKING

Reading input files...
Source 1 records: 8
Source 2 records: 12
Source 3 records: 12

Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 2 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 3 columns:
['entity_id', 'business_name', 'business_address', 'country']

Preparing data...
✓ Data preparation complete

Generating Source 2 candidates

Building token index...
Token index entries: 67
Building country index...
Country index entries: 2
Source 2 candidate pairs: 15

Generating Source 3 candid